# 🎯 [C202_M01] 협대역 전술 무선망(VHF/UHF) 대역폭 제약 조건 분석 및 전술 메시지 구조 정의
### Google Colab 실습 환경 (지휘통제(C2) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C202/c202_m01_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 2) |
| **미션 ID** | C202_M01 |
| **미션 주제** | 협대역 전술 무선망(VHF/UHF) 대역폭 제약 조건 분석 및 전술 메시지 구조 정의 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 전자전 재밍과 통신 장비 부족으로 협대역 무선망의 대역폭이 크게 줄어드는 상황이 잦았고, 이때 적 접촉 보고나 의무후송 요청 같은 긴급 전문이 행정 전문 뒤에 묶여 늦게 전달되는 문제가 지휘통제의 약점이 되었다. 본 프로젝트에서는 가상 협대역 링크(2.4kbps)와 가상 전술 전문을 코드로 직접 만들어, 전문 구조 정의, 패킷 파싱·압축 전처리, 긴급도 기반 우선순위 큐, 채널 상태에 따른 지능형 재전송, 링크 신뢰도 산출, 통신 관제 화면, 통신관 HITL 통제, 대역폭 급감 스트레스 시험, AAR 재조정, 최종 통합까지 메시지 우선순위 큐·재전송 프로토타입을 단계별로 구현한다. 최종 목표는 대역폭이 제한된 상황에서도 FLASH·IMMEDIATE 전문을 기한 안에 전달하고, 어떤 전문도 잃어버리지 않으며, 순서 조정의 최종 판단은 통신관이 내리도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c202_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c202.py` | 전처리, 우선순위 큐, 재전송 제어, 링크 신뢰도, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c202_data_spec.md`, `docs/c202_hitl_guide.md`, `docs/c202_aar_report.md`, `docs/c202_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c202_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c202_stress_test.py` | 대역폭 90% 급감 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c202_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 협대역 전술 무선망(VHF/UHF) 대역폭 제약 조건 분석 및 전술 메시지 구조 정의
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

협대역 전술 무전망(VHF/UHF)의 전송 대역폭 제약과 전술 전문(Message) 구조를 분석한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **협대역 전술 무선망의 대역폭 제약이 전문 전송시간에 주는 영향을 분석하고, 전문 헤더·긴급도·링크 조건·전술 파라미터를 담은 데이터 스키마와 검증기를 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **전문 유형(접촉 보고, 의무후송 요청, 상황 보고, 위치 보고, 보급 요청, 행정)별 평균 크기를 1.2~16kbps 링크에서 보낼 때 걸리는 시간을 표로 정리하고 규격화하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `data/schema/c202_input_schema.json 및 docs/c202_data_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지휘통제(C2) 실무 및 러시아-우크라이나 전쟁 전장 환경과의 군사적 부합성

#### 미션 PASS 조건

- 정량 기준: 스키마 필드 정의 완성도 100%, 샘플 데이터 검증 오류 0건


- 진행 기준: 입력 데이터 유효성 검증 스크립트 통과 및 명세서 문서화 완료


- 검증 방법: JSON Schema 검증 도구 및 샘플 데이터 파싱 단위 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""협대역 전술 무선망(VHF/UHF)의 대역폭 제약을 전문 유형별 전송시간으로 분석하고, 전술 전문의 헤더·긴급도·링크·전술 파라미터 스키마를 정의해 가상 샘플 데이터를 검증한다."""
import json
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 전술 전문(메시지) 생성기 (실제 부대·통신망·제원 정보 없음)
# 긴급도 4단계(FLASH/IMMEDIATE/PRIORITY/ROUTINE)와 전문 유형별 크기를 가진 전문을 만든다.
# ---------------------------------------------------------------------------
PREC = ["FLASH", "IMMEDIATE", "PRIORITY", "ROUTINE"]
RANK = {p: i for i, p in enumerate(PREC)}
DEADLINE = {"FLASH": 10, "IMMEDIATE": 30, "PRIORITY": 120, "ROUTINE": 600}   # 가상 전달 목표 시간(초)
MSG_TYPES = {   # 유형: (긴급도 후보, 확률, 본문 크기 범위 bytes, 발생 비율)
    "CONTACT": (["FLASH", "IMMEDIATE"], [0.7, 0.3], (120, 300), 0.08),
    "MEDEVAC": (["FLASH", "IMMEDIATE"], [0.4, 0.6], (150, 350), 0.05),
    "SITREP": (["IMMEDIATE", "PRIORITY"], [0.2, 0.8], (600, 2500), 0.20),
    "POSREP": (["PRIORITY", "ROUTINE"], [0.5, 0.5], (60, 120), 0.35),
    "LOGREQ": (["PRIORITY", "ROUTINE"], [0.4, 0.6], (300, 1200), 0.14),
    "ADMIN": (["ROUTINE"], [1.0], (1000, 5000), 0.18),
}


def make_messages(duration=600, rate=0.12, seed=7, t_start=0.0, prefix="M"):
    """duration초 동안 평균 rate건/초로 발생하는 가상 전문 목록."""
    rng = np.random.default_rng(seed)
    n = int(rng.poisson(rate * duration))
    times = np.sort(rng.uniform(t_start, t_start + duration, n))
    types = list(MSG_TYPES)
    msgs = []
    for i, t0 in enumerate(times):
        ty = types[rng.choice(len(types), p=[MSG_TYPES[k][3] for k in types])]
        cands, p, (lo, hi), _ = MSG_TYPES[ty]
        msgs.append({"id": f"{prefix}{i:04d}", "t0": float(t0), "type": ty, "prec": str(rng.choice(cands, p=p)),
                     "size": int(rng.integers(lo, hi + 1)), "src": f"U{int(rng.integers(1, 9)):02d}"})
    return msgs


class Link:
    """[교육용가상자료] 가상 협대역 무선 링크(기본 2.4kbps). 초 단위 상태(good/bad/jam)에 따라
    비트오류율이 바뀌고(Gilbert-Elliott 모델), 구간별 대역폭 감소도 줄 수 있다."""
    BER = {"good": 1e-5, "bad": 2e-4, "jam": 1.2e-3}

    def __init__(self, horizon=4000, bw_bps=2400, seed=1, p_gb=0.02, p_bg=0.15, jam=(), bw_drop=(), rtt=(1.2, 0.4)):
        rng = np.random.default_rng(seed)
        st, s = [], "good"
        for _ in range(int(horizon) + 1):
            s = ("bad" if rng.random() < p_gb else "good") if s == "good" else ("good" if rng.random() < p_bg else "bad")
            st.append(s)
        for a, b in jam:
            for k in range(int(a), min(int(b), len(st))):
                st[k] = "jam"
        self.states, self.bw_bps, self.bw_drop, self.rtt_par = st, bw_bps, list(bw_drop), rtt

    def state(self, t):
        return self.states[min(int(t), len(self.states) - 1)]

    def bw(self, t):
        f = 1.0
        for a, b, fac in self.bw_drop:
            if a <= t < b:
                f = fac
        return self.bw_bps * f

    def p_loss(self, nbytes, t):
        return 1 - (1 - self.BER[self.state(t)]) ** (8 * nbytes)

    def rtt(self, rng):
        return self.rtt_par[0] + rng.exponential(self.rtt_par[1])
# ---------------------------------------------------------------------------

# 1) 대역폭 제약 분석: 링크 속도별 전문 1건 전송시간(헤더 32bytes/조각 256bytes 기준, 재전송 제외)
print("[대역폭별 전문 1건 전송시간(초) - 가상 링크]")
rows = []
for ty, (_, _, (lo, hi), _) in MSG_TYPES.items():
    size = (lo + hi) / 2
    nbytes = size + np.ceil(size / 256) * 32
    rows.append({"유형": ty, "평균 크기(B)": int(size), **{f"{bw / 1000:g}kbps": round(nbytes * 8 / bw, 1) for bw in (1200, 2400, 9600, 16000)}})
print(pd.DataFrame(rows).to_string(index=False))
print("-> 2.4kbps에서 행정전문 1건이 약 10초 이상 채널을 점유한다. FLASH 전달 목표(10초)를 지키려면 우선순위 큐와 조각 단위 전송이 필요하다.")

# 2) 스키마 정의: 전문 헤더, 링크 조건, 전술 파라미터(기상·위치오차·통신지연)
SCHEMA = {
    "required": ["msg_id", "precedence", "msg_type", "src", "created_s", "payload_bytes", "link", "tactical"],
    "fields": {
        "msg_id": {"type": str, "pattern": r"^M\d{4}$"},
        "precedence": {"type": str, "enum": PREC},
        "msg_type": {"type": str, "enum": list(MSG_TYPES)},
        "src": {"type": str, "pattern": r"^U\d{2}$"},
        "created_s": {"type": (int, float), "min": 0},
        "payload_bytes": {"type": int, "min": 1, "max": 8000},
    },
    "link": {
        "band": {"type": str, "enum": ["VHF", "UHF"]},
        "bandwidth_bps": {"type": int, "enum": [1200, 2400, 9600, 16000]},
        "snr_db": {"type": (int, float), "min": -10, "max": 40},
        "mtu_bytes": {"type": int, "min": 32, "max": 1024},
    },
    "tactical": {
        "weather": {"type": str, "enum": ["clear", "rain", "fog", "snow", "night"]},
        "position_error_m": {"type": (int, float), "min": 0, "max": 500},
        "comm_delay_ms": {"type": (int, float), "min": 0, "max": 30000},
        "relay_hops": {"type": int, "min": 0, "max": 5},
    },
}


def check(name, value, rule):
    """값 하나를 규칙(type, enum, min, max, pattern)으로 검사해 오류 목록을 돌려준다."""
    # [TODO : 학생 실습 1] 협대역 전술 무선망(VHF/UHF) 대역폭 제약 조건 분석 및 전술 메시지 구조 정의 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def validate(rec):
    # [TODO : 학생 실습 1] 협대역 전술 무선망(VHF/UHF) 대역폭 제약 조건 분석 및 전술 메시지 구조 정의 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
rng = np.random.default_rng(1)
records = [{"msg_id": m["id"], "precedence": m["prec"], "msg_type": m["type"], "src": m["src"],
            "created_s": round(m["t0"], 1), "payload_bytes": m["size"],
            "link": {"band": str(rng.choice(["VHF", "UHF"])), "bandwidth_bps": 2400, "snr_db": float(round(rng.uniform(3, 25), 1)), "mtu_bytes": 256},
            "tactical": {"weather": str(rng.choice(["clear", "rain", "fog", "snow", "night"])), "position_error_m": float(round(rng.uniform(3, 50), 1)),
                         "comm_delay_ms": int(rng.integers(300, 3000)), "relay_hops": int(rng.integers(0, 3))}}
           for m in make_messages(400, 0.12, seed=3)[:20]]
ok = sum(1 for r in records if not validate(r))
print(f"\n정상 샘플 {len(records)}건 중 검증 통과 {ok}건, 오류 {len(records) - ok}건")

# 4) 오류 주입 시험
bad = json.loads(json.dumps(records[0]))
bad["msg_id"] = "MSG-1"
bad["precedence"] = "URGENT"
bad["link"]["bandwidth_bps"] = 3000
del bad["tactical"]["comm_delay_ms"]
bad["tactical"]["relay_hops"] = 9
print("[오류 주입 결과]")
for e in validate(bad):
    print(" -", e)
flash_big = {**records[0], "precedence": "FLASH", "payload_bytes": 4000}
print(" -", validate(flash_big)[0])

# 5) 스키마 명세 요약
rows = []
for g in ("fields", "link", "tactical"):
    for k, r in SCHEMA[g].items():
        t = r["type"].__name__ if isinstance(r["type"], type) else "number"
        rows.append(("기본" if g == "fields" else g, k, t, str({x: y for x, y in r.items() if x != "type"})[:58]))
print("\n[데이터 명세 요약]")
print(pd.DataFrame(rows, columns=["그룹", "필드", "자료형", "규칙"]).to_string(index=False))
print("\n긴급도별 전달 목표(초):", DEADLINE)


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[대역폭별 전문 1건 전송시간(초) - 가상 링크]
     유형  평균 크기(B)  1.2kbps  2.4kbps  9.6kbps  16kbps
CONTACT       210      1.6      0.8      0.2     0.1
MEDEVAC       250      1.9      0.9      0.2     0.1
 SITREP      1550     11.8      5.9      1.5     0.9
 POSREP        90      0.8      0.4      0.1     0.1
 LOGREQ       750      5.6      2.8      0.7     0.4
  ADMIN      3000     22.6     11.3      2.8     1.7
-> 2.4kbps에서 행정전문 1건이 약 10초 이상 채널을 점유한다. FLASH 전달 목표(10초)를 지키려면 우선순위 큐와 조각 단위 전송이 필요하다.

정상 샘플 20건 중 검증 통과 20건, 오류 0건
[오류 주입 결과]
 - msg_id: 형식 불일치 'MSG-1'
 - precedence: 허용되지 않은 값 'URGENT'
 - link.bandwidth_bps: 허용되지 않은 값 '3000'
 - tactical.comm_delay_ms: 필수 필드 누락
 - tactical.relay_hops: 최대값 5 초과
 - 업무 규칙: FLASH 전문은 2,000bytes 이하로 작성

[데이터 명세 요약]
      그룹               필드    자료형                                                         규칙
      기본           msg_id    str                                   {'pattern': '^M\\d{4}$'}
      기본       precedence    str    {'enum': ['FLASH', 'IMMEDIATE', 'PRIORITY', 'ROUTINE']}
      기본         msg_type    str {'enum': ['CONTACT', 'MEDEVAC', 'SITREP', 'POSREP', 'LOGRE
      기본              src    str                                   {'pattern': '^U\\d{2}$'}
      기본        created_s number                                                 {'min': 0}
      기본    payload_bytes    int                                    {'min': 1, 'max': 8000}
    link             band    str                                   {'enum': ['VHF', 'UHF']}
    link    bandwidth_bps    int                        {'enum': [1200, 2400, 9600, 16000]}
    link           snr_db number                                    {'min': -10, 'max': 40}
    link        mtu_bytes    int                                   {'min': 32, 'max': 1024}
tactical          weather    str        {'enum': ['clear', 'rain', 'fog', 'snow', 'night']}
tactical position_error_m number                                     {'min': 0, 'max': 500}
tactical    comm_delay_ms number                                   {'min': 0, 'max': 30000}
tactical       relay_hops    int                                       {'min': 0, 'max': 5}

긴급도별 전달 목표(초): {'FLASH': 10, 'IMMEDIATE': 30, 'PRIORITY': 120, 'ROUTINE': 600}
```
</details>
